# Depth Estimation — Visual Results

Side-by-side comparison: **RGB | GT Depth | Baseline | LoRA**

- 10 samples from NYU Depth v2 test set
- 10 samples from DIODE val set (5 indoor + 5 outdoor)

All depth maps are colourised with the **turbo** colormap, normalised to the GT range per image so colours are directly comparable across columns.

## 0 — Config
Edit the paths below before running.

In [ ]:
import os, sys

# ── Edit these ────────────────────────────────────────────────────────────────
REPO_ROOT     = os.path.abspath('.')          # path to 4m-peft/ (run jupyter from repo root)
NYU_ROOT      = '/scratch-shared/tesr125338/nyu_raw/test'
DIODE_ROOT    = '/scratch-shared/tesr125338/diode/val'
LORA_WEIGHTS  = 'output/depth/nyu_rgb_r8/lora_weights-best.pth'
LORA_RANK     = 8           # must match training config (nyu_rgb_r8 → rank=8)
LORA_MODULES  = 'attention' # must match training config

FM_MODEL   = 'EPFL-VILAB/4M-21_XL'
TOK_DEPTH  = 'EPFL-VILAB/4M_tokenizers_depth_8k_224-448'
TOK_TEXT   = 'ml-4m/fourm/utils/tokenizer/trained/text_tokenizer_4m_wordpiece_30k.json'

N_SAMPLES    = 10    # images per dataset
DIVAE_STEPS  = 50
SEED         = 42
DEVICE       = 'cuda'
OUTPUT_DIR   = os.path.join(REPO_ROOT, 'output', 'viz_depth')
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'ml-4m'))
os.chdir(REPO_ROOT)
print(f'Repo root : {REPO_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')

## 1 — Imports

In [ ]:
import copy
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch
from PIL import Image
from tokenizers import Tokenizer
from torchvision.transforms.functional import center_crop

from fourm.models.fm import FM
from fourm.vq.vqvae import DiVAE
from fourm.models.generate import (
    GenerationSampler, build_chained_generation_schedules,
    init_empty_target_modality, init_full_input_modality,
)
from fourm.data.modality_info import MODALITY_INFO
from fourm.data.modality_transforms import RGBTransform
from fourm.utils.plotting_utils import decode_tok_depth
from fourm.models.lora_utils import inject_trainable_LoRA, get_LoRA_module_names

DEPTH_GEN_PARAMS = {
    'tokens_per_target': 196, 'autoregression_scheme': 'roar',
    'decoding_steps': 1, 'token_decoding_schedule': 'linear',
    'temp': 0.01, 'temp_schedule': 'constant',
    'cfg_scale': 2.0, 'cfg_schedule': 'constant',
}

device  = torch.device(DEVICE)
dtype_t = torch.bfloat16
print('Imports OK')

## 2 — Load models

In [ ]:
# ── Baseline FM ───────────────────────────────────────────────────────────────
print('Loading baseline FM ...')
fm_base = FM.from_pretrained(FM_MODEL).to(device=device, dtype=dtype_t).eval()
sampler_base = GenerationSampler(fm_base)
print(f'  Baseline ready  ({sum(p.numel() for p in fm_base.parameters())/1e6:.0f}M params)')

# ── LoRA FM ───────────────────────────────────────────────────────────────────
lora_path = os.path.join(REPO_ROOT, LORA_WEIGHTS)
if os.path.exists(lora_path):
    print(f'Loading LoRA model ...')
    fm_lora = FM.from_pretrained(FM_MODEL)

    # CRITICAL: inject LoRA architecture BEFORE loading weights.
    # Without this, load_state_dict(strict=False) silently drops all lora_A/lora_B
    # keys because they don't exist in the base model — giving identical output to baseline.
    target_modules = get_LoRA_module_names(LORA_MODULES)
    inject_trainable_LoRA(fm_lora, rank=LORA_RANK, scale=1.0,
                          target_replace_modules=target_modules)
    print(f'  LoRA architecture injected (rank={LORA_RANK}, modules={target_modules})')

    lora_sd = torch.load(lora_path, map_location='cpu')
    missing, unexpected = fm_lora.load_state_dict(lora_sd, strict=False)
    print(f'  Loaded {len(lora_sd)} LoRA tensors  (missing={len(missing)}, unexpected={len(unexpected)})')
    if missing:
        print(f'  WARNING missing keys (first 3): {missing[:3]}')

    fm_lora = fm_lora.to(device=device, dtype=dtype_t).eval()
    sampler_lora = GenerationSampler(fm_lora)
    LORA_AVAILABLE = True
    print('  LoRA model ready.')
else:
    print(f'  WARNING: LoRA weights not found at {lora_path}')
    print('  → LoRA column will show N/A. Run training first.')
    sampler_lora = None
    LORA_AVAILABLE = False

# ── DiVAE + text tokenizer ────────────────────────────────────────────────────
print('Loading DiVAE ...')
tok_depth = DiVAE.from_pretrained(TOK_DEPTH).to(device=device).eval()
tok_text  = Tokenizer.from_file(os.path.join(REPO_ROOT, TOK_TEXT))
print('All models loaded.')

## 3 — Helper functions

In [ ]:
def _center_crop_np(arr, size):
    h, w = arr.shape[:2]
    return arr[(h-size)//2:(h-size)//2+size, (w-size)//2:(w-size)//2+size]


def load_rgb_pil_and_tensor(rgb_path):
    """Returns (display_pil 224x224 RGB, model_tensor 1x3x224x224)."""
    t = RGBTransform(imagenet_default_mean_and_std=True)
    img = t.load(str(rgb_path))
    img = t.preprocess(img)
    s   = min(img.size)
    img = center_crop(img, (s, s)).resize((224, 224), Image.BILINEAR)
    tensor = t.postprocess(img).unsqueeze(0).to(device=device, dtype=dtype_t)
    return img, tensor


def load_gt_nyu(depth_path, max_depth=10.0):
    depth = np.load(str(depth_path)).squeeze().astype(np.float32)
    h, w  = depth.shape
    depth = _center_crop_np(depth, min(h, w))
    depth = cv2.resize(depth, (224, 224), interpolation=cv2.INTER_LINEAR)
    valid = (depth > 0) & (depth <= max_depth)
    return depth, valid


def load_gt_diode(depth_path, mask_path, max_depth=300.0):
    depth = np.load(str(depth_path)).squeeze().astype(np.float32)
    mask  = np.load(str(mask_path)).squeeze()
    h, w  = depth.shape
    s     = min(h, w)
    depth = _center_crop_np(depth, s)
    mask  = _center_crop_np(mask, s)
    depth = cv2.resize(depth, (224, 224), interpolation=cv2.INTER_LINEAR)
    mask  = cv2.resize(mask.astype(np.float32), (224, 224), interpolation=cv2.INTER_NEAREST)
    valid = (mask > 0) & (depth > 0) & (depth <= max_depth)
    return depth, valid


def run_inference(sampler, rgb_tensor, seed):
    """Returns raw z-score depth map (224,224)."""
    ntoks    = DEPTH_GEN_PARAMS['tokens_per_target']
    schedule = build_chained_generation_schedules(
        cond_domains=['rgb@224'], target_domains=['tok_depth@224'],
        tokens_per_target=[ntoks],
        autoregression_schemes=[DEPTH_GEN_PARAMS['autoregression_scheme']],
        decoding_steps=[DEPTH_GEN_PARAMS['decoding_steps']],
        token_decoding_schedules=[DEPTH_GEN_PARAMS['token_decoding_schedule']],
        temps=[DEPTH_GEN_PARAMS['temp']], temp_schedules=[DEPTH_GEN_PARAMS['temp_schedule']],
        cfg_scales=[DEPTH_GEN_PARAMS['cfg_scale']], cfg_schedules=[DEPTH_GEN_PARAMS['cfg_schedule']],
        cfg_grow_conditioning=True,
    )
    eos_id = tok_text.token_to_id('[EOS]')
    sd = {'rgb@224': {'tensor': rgb_tensor}}
    sd = init_full_input_modality(sd, MODALITY_INFO, 'rgb@224', device, eos_id=eos_id)
    sd = init_empty_target_modality(sd, MODALITY_INFO, 'tok_depth@224', 1, ntoks, device)
    out = sampler.generate(sd, schedule, text_tokenizer=tok_text, verbose=False, seed=seed)
    raw = decode_tok_depth(out, {'tok_depth': tok_depth}, key='tok_depth@224',
                           image_size=224, patch_size=16, t=DIVAE_STEPS,
                           verbose=False, cmap=None)
    return raw[0].astype(np.float32) if raw.ndim == 3 else raw.astype(np.float32)


def scale_shift_align(pred, gt, valid):
    """Least-squares affine alignment: pred → metric space."""
    p = pred[valid].reshape(-1, 1).astype(np.float64)
    g = gt[valid].reshape(-1).astype(np.float64)
    A = np.hstack([p, np.ones_like(p)])
    scale, shift = np.linalg.lstsq(A, g, rcond=None)[0]
    return np.clip(scale * pred + shift, 1e-3, None).astype(np.float32)


def compute_abs_rel(pred_m, gt_m, valid):
    p, g = pred_m[valid].astype(np.float64), gt_m[valid].astype(np.float64)
    return float(np.mean(np.abs(p - g) / g))


def depth_to_rgb(depth, valid, vmin=None, vmax=None, cmap='turbo'):
    """Convert a depth map to a colourised (224,224,3) uint8 image.
    Invalid pixels are shown as dark grey.
    vmin/vmax set a shared colour range — pass GT range for comparability.
    """
    d = depth.copy()
    d[~valid] = np.nan
    if vmin is None:
        vmin = float(np.nanmin(d))
    if vmax is None:
        vmax = float(np.nanmax(d))
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cm   = plt.get_cmap(cmap)
    rgba = cm(norm(d))                     # (H,W,4) float [0,1]
    rgb  = (rgba[:, :, :3] * 255).astype(np.uint8)
    rgb[~valid] = 40                        # dark grey for invalid pixels
    return rgb


print('Helpers defined.')

## 4 — Sample collection

In [ ]:
def collect_nyu(root, n):
    root = Path(root)
    pairs = []
    for rgb_p in sorted((root / 'rgb').glob('*.png')) + sorted((root / 'rgb').glob('*.jpg')):
        d = root / 'depth' / f'{rgb_p.stem}.npy'
        if d.exists():
            pairs.append({'rgb': rgb_p, 'depth': d, 'mask': None,
                          'split': 'nyu', 'name': rgb_p.stem})
    random.seed(SEED)
    return random.sample(pairs, min(n, len(pairs)))


def collect_diode(root, n):
    root = Path(root)
    samples = {'indoors': [], 'outdoor': []}
    for split in ['indoors', 'outdoor']:
        sd = root / split
        if not sd.exists():
            continue
        for scene in sorted(sd.iterdir()):
            for scan in sorted(scene.iterdir()):
                for rgb_p in sorted(scan.glob('*.png')):
                    stem = rgb_p.stem
                    dp   = scan / f'{stem}_depth.npy'
                    mp   = scan / f'{stem}_depth_mask.npy'
                    if dp.exists() and mp.exists():
                        samples[split].append(
                            {'rgb': rgb_p, 'depth': dp, 'mask': mp,
                             'split': split, 'name': f'{split}/{scene.name}/{stem}'})
    random.seed(SEED)
    half = n // 2
    return (random.sample(samples['indoors'],  min(half, len(samples['indoors']))) +
            random.sample(samples['outdoor'],  min(n - half, len(samples['outdoor']))))


nyu_samples   = collect_nyu(NYU_ROOT, N_SAMPLES)
diode_samples = collect_diode(DIODE_ROOT, N_SAMPLES)
print(f'NYU   samples : {len(nyu_samples)}')
print(f'DIODE samples : {len(diode_samples)} ({sum(s["split"]=="indoors" for s in diode_samples)} indoor, {sum(s["split"]=="outdoor" for s in diode_samples)} outdoor)')

## 5 — Core visualization function

In [ ]:
def visualize_samples(samples, dataset_name, load_gt_fn, max_depth, save_path):
    """
    Renders a grid: N rows × 4 columns (RGB | GT | Baseline | LoRA).
    All depth columns share the GT colour range per row.
    AbsRel is shown in the subtitle of each depth column.
    """
    n_cols = 4 if LORA_AVAILABLE else 3
    col_labels = ['RGB', 'GT Depth', 'Baseline', 'LoRA'] if LORA_AVAILABLE \
                 else ['RGB', 'GT Depth', 'Baseline']

    fig, axes = plt.subplots(
        nrows=len(samples), ncols=n_cols,
        figsize=(n_cols * 3, len(samples) * 3),
        squeeze=False,
    )

    # Column headers
    for col, label in enumerate(col_labels):
        axes[0, col].set_title(label, fontsize=13, fontweight='bold', pad=8)

    for row, s in enumerate(samples):
        print(f'  [{row+1}/{len(samples)}] {s["name"]}', flush=True)

        # ── Load data ──────────────────────────────────────────────────────────
        rgb_pil, rgb_tensor = load_rgb_pil_and_tensor(s['rgb'])
        gt_depth, valid     = load_gt_fn(s)

        if valid.sum() < 100:
            for ax in axes[row]:
                ax.axis('off')
                ax.text(0.5, 0.5, 'skip\n(no valid px)', ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='red')
            continue

        # Shared colour range: GT valid pixel range
        gt_valid_vals = gt_depth[valid]
        vmin, vmax    = float(gt_valid_vals.min()), float(gt_valid_vals.max())

        # ── Run inference ──────────────────────────────────────────────────────
        with torch.no_grad():
            raw_base = run_inference(sampler_base, rgb_tensor, seed=SEED + row)
        pred_base = scale_shift_align(raw_base, gt_depth, valid)
        ar_base   = compute_abs_rel(pred_base, gt_depth, valid)

        if LORA_AVAILABLE:
            with torch.no_grad():
                raw_lora = run_inference(sampler_lora, rgb_tensor, seed=SEED + row)
            pred_lora = scale_shift_align(raw_lora, gt_depth, valid)
            ar_lora   = compute_abs_rel(pred_lora, gt_depth, valid)

        # ── Colourize depth maps ───────────────────────────────────────────────
        gt_rgb   = depth_to_rgb(gt_depth,   valid, vmin, vmax)
        base_rgb = depth_to_rgb(pred_base,  valid, vmin, vmax)
        if LORA_AVAILABLE:
            lora_rgb = depth_to_rgb(pred_lora, valid, vmin, vmax)

        # ── Row label ──────────────────────────────────────────────────────────
        split_tag = f'[{s["split"]}]  ' if s['split'] != 'nyu' else ''
        axes[row, 0].set_ylabel(f'{split_tag}{Path(s["name"]).name}',
                                fontsize=8, rotation=0, labelpad=80,
                                ha='right', va='center')

        # ── Column 0: RGB ──────────────────────────────────────────────────────
        axes[row, 0].imshow(rgb_pil)
        axes[row, 0].axis('off')

        # ── Column 1: GT ───────────────────────────────────────────────────────
        axes[row, 1].imshow(gt_rgb)
        axes[row, 1].set_xlabel(f'range [{vmin:.1f}, {vmax:.1f}]m',
                                fontsize=7)
        axes[row, 1].axis('off')

        # ── Column 2: Baseline ─────────────────────────────────────────────────
        axes[row, 2].imshow(base_rgb)
        axes[row, 2].set_xlabel(f'AbsRel = {ar_base:.3f}', fontsize=8,
                                color='#1a6bb5', fontweight='bold')
        axes[row, 2].axis('off')

        # ── Column 3: LoRA ─────────────────────────────────────────────────────
        if LORA_AVAILABLE:
            better = ar_lora < ar_base
            color  = '#2a9d2a' if better else '#c0392b'
            axes[row, 3].imshow(lora_rgb)
            axes[row, 3].set_xlabel(
                f'AbsRel = {ar_lora:.3f}  ({"▲ better" if better else "▼ worse"})',
                fontsize=8, color=color, fontweight='bold')
            axes[row, 3].axis('off')

    fig.suptitle(f'Depth Estimation — {dataset_name}\n'
                 f'Colour range = GT range per row | Alignment = scale-shift (affine-invariant)',
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'  Saved → {save_path}')
    plt.show()


print('Visualization function ready.')

## 6 — NYU Depth v2 results

In [ ]:
print('Generating NYU visualizations ...')

def _load_gt_nyu(s):
    return load_gt_nyu(s['depth'], max_depth=10.0)

visualize_samples(
    samples      = nyu_samples,
    dataset_name = 'NYU Depth v2 — test set',
    load_gt_fn   = _load_gt_nyu,
    max_depth    = 10.0,
    save_path    = os.path.join(OUTPUT_DIR, 'nyu_depth_comparison.png'),
)

## 7 — DIODE results

In [ ]:
print('Generating DIODE visualizations ...')

def _load_gt_diode(s):
    return load_gt_diode(s['depth'], s['mask'], max_depth=300.0)

visualize_samples(
    samples      = diode_samples,
    dataset_name = 'DIODE val — indoor + outdoor',
    load_gt_fn   = _load_gt_diode,
    max_depth    = 300.0,
    save_path    = os.path.join(OUTPUT_DIR, 'diode_depth_comparison.png'),
)

## 8 — Summary metrics table

In [ ]:
import json

def load_summary(run_name):
    path = os.path.join(REPO_ROOT, 'output', 'eval_depth', run_name, 'results.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)['summary']

runs = {
    'NYU   Baseline': load_summary('nyu_baseline'),
    'NYU   LoRA':     load_summary('nyu_lora'),
    'DIODE Baseline': load_summary('diode_baseline'),
    'DIODE LoRA':     load_summary('diode_lora'),
}

print(f'{'Run':<20}  {'AbsRel':>8}  {'RMSE':>8}  {'δ<1.25':>8}  {'δ<1.25²':>8}  {'L1':>8}  N')
print('-' * 75)
for label, s in runs.items():
    if s is None:
        print(f'{label:<20}  (results not found — run 09_eval_depth.sh first)')
        continue
    m = s['metrics_all'].get('all', {})
    print(f'{label:<20}  '
          f'{m.get("abs_rel",float("nan")):>8.4f}  '
          f'{m.get("rmse",float("nan")):>8.4f}  '
          f'{m.get("delta1",float("nan")):>8.4f}  '
          f'{m.get("delta2",float("nan")):>8.4f}  '
          f'{m.get("l1_norm",float("nan")):>8.4f}  '
          f'{s["n_valid"]}')